# https://blast.ncbi.nlm.nih.gov/doc/blast-help/downloadblastdata.html#blast-executables

In [ ]:
# Instalação via apt

sudo apt update
sudo apt install ncbi-blast+

In [ ]:
# Ou instalação manual

wget -c https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ncbi-blast-2.17.0+-x64-linux.tar.gz

# Extrair
sudo tar -xzf ncbi-blast-2.17.0+-x64-linux.tar.gz

In [ ]:
# Instalação via micromamba - Recomendada

micromamba create -n Blast
micromamba activate Blast
micromamba install -c bioconda -c conda-forge blast

In [ ]:
# Verificar instalação
blastn -version
blastp -version
blastx -version

In [ ]:
# Baixar e enviar banco de dados que sera utilizado para o servidor 
## https://tncentral.ncc.unesp.br/data_download
### TNCentral+ISFinder Database - Nucleotide FASTA file
scp C:\Users\vitor\Downloads\nc_tncentral_isfinder.fa.zip victorbaca@10.20.34.31:/mnt/dados/victor_baca/nc_tncentral_isfinder

# Ir ao diretório onde baixou o arquivo fasta do seu banco de dados
cd /mnt/dados/victor_baca/nc_tncentral_isfinder
unzip nc_tncentral_isfinder.fa.zip # Descomprimir o arquivo zip
rm -rf *.fa.* # Remover arquivos desnecessários

makeblastdb -in tncentral_isfinder.fa -dbtype nucl -parse_seqids -title "TnCentral TE Database" -out TnCentral_db

In [ ]:
# Executar o BLASTn (Blast para sequências de nucleotídeos)

# Maneira manualmente de rodar o comando abaixo, alterando apenas o caminho dos genomas e do output

## Para facilitar a localização do genoma que deseja analisar, utilize o comando find:
### Encontrar o genoma isolado que deseja analisar
find path -iname ID_NCBI*

### EX:
find /mnt/dados/victor_baca/genoma -iname GCF_002812445.1_ASM281244v1*

blastn -query /mnt/dados/victor_baca/genoma/GCF_002812445.1_ASM281244v1_genomic.fna -db /mnt/dados/victor_baca/nc_tncentral_isfinder/TnCentral_db -out SGEHI2015-95.txt -evalue 1e-5 -outfmt "6 qseqid sseqid pident qcovs length mismatch gapopen qstart qend sstart send sstrand evalue bitscore qlen slen sframe"

In [ ]:
# Automatizando o processo para vários genomas

## Criando script para rodar vários genomas de uma vez
nano blast_genomes.sh

## Script #########################################################################################################################################
#!/bin/bash

# Diretório contendo os genomas
GENOMES_DIR="/mnt/dados/victor_baca/genoma"
# Banco de dados BLAST
BLAST_DB="/mnt/dados/victor_baca/nc_tncentral_isfinder/TnCentral_db"
# Diretório de saída
OUTPUT_DIR="/mnt/dados/victor_baca/outputs/BLAST"

# Verifica se o diretório de genomas existe
if [ ! -d "$GENOMES_DIR" ]; then
    echo "Erro: Diretório $GENOMES_DIR não encontrado!"
    exit 1
fi

# Verifica se o banco de dados existe
if [ ! -f "${BLAST_DB}.nin" ] && [ ! -f "${BLAST_DB}.nhr" ]; then
    echo "Erro: Banco de dados BLAST não encontrado em $BLAST_DB"
    echo "Verifique se os arquivos .nin, .nhr, .nsq existem"
    exit 1
fi

# Cria diretório de saída se não existir
mkdir -p "$OUTPUT_DIR"

# Contador para acompanhar o progresso
total_files=0
processed_files=0

# Conta o total de arquivos .fna
total_files=$(find "$GENOMES_DIR" -name "*.fna" -type f | wc -l)

if [ $total_files -eq 0 ]; then
    echo "Nenhum arquivo .fna encontrado em $GENOMES_DIR"
    exit 1
fi

echo "Encontrados $total_files arquivos .fna para processar"
echo "Outputs serão salvos em: $OUTPUT_DIR"
echo "Iniciando análises BLAST..."
echo "----------------------------------------"

# Loop através de todos os arquivos .fna no diretório
find "$GENOMES_DIR" -name "*.fna" -type f | while read -r genome_file; do
    # Extrai o nome base do arquivo (sem extensão e caminho)
    basename_file=$(basename "$genome_file" .fna)
    
    # Define o arquivo de saída
    output_file="${OUTPUT_DIR}/${basename_file}_blast.txt"
    
    # Incrementa contador
    ((processed_files++))
    
    echo "[$processed_files/$total_files] Processando: $basename_file"
    echo "  Input: $genome_file"
    echo "  Output: $output_file"
    
    # Executa BLAST
    blastn \
        -query "$genome_file" \
        -db "$BLAST_DB" \
        -out "$output_file" \
        -evalue 1e-5 \
        -outfmt "6 qseqid sseqid pident qcovs length mismatch gapopen qstart qend sstart send sstrand evalue bitscore qlen slen sframe" \
    
    # Verifica se o BLAST foi executado com sucesso
    if [ $? -eq 0 ]; then
        echo "  ✓ Concluído com sucesso"
        # Mostra quantos hits foram encontrados
        hit_count=$(wc -l < "$output_file")
        echo "  → $hit_count hits encontrados"
    else
        echo "  ✗ Erro na execução do BLAST"
    fi
    
    echo "----------------------------------------"
done

echo "Análise BLAST concluída para todos os genomas!"
echo "Arquivos de saída salvos em: $OUTPUT_DIR"
###################################################################################################################################################

chmod +x blast_genomes.sh
./blast_genomes.sh

In [ ]:
# Criar um script para converter os arquivos .txt gerados pelo BLAST em .gff
nano blast_to_gff.py 
## Copiar o script abaixo

## Script #########################################################################################################################################

#!/usr/bin/env python3

import os
import sys
import argparse
from pathlib import Path

def parse_blast_line(line):
    fields = line.strip().split('\t')
    
    if len(fields) < 14:
        return None
    
    try:
        # Estrutura correta do arquivo BLAST customizado
        qseqid = fields[0]          # Query Sequence ID
        sseqid = fields[1]          # Subject Sequence ID
        pident = float(fields[2])   # Percentage of identical matches
        qcovs = float(fields[3])    # Query Coverage Per Subject
        length = int(fields[4])     # Alignment length
        mismatch = int(fields[5])   # Number of mismatches
        gapopen = int(fields[6])    # Number of gap openings
        qstart = int(fields[7])     # Start of alignment in query
        qend = int(fields[8])       # End of alignment in query
        sstart = int(fields[9])     # Start of alignment in subject
        send = int(fields[10])      # End of alignment in subject
        sstrand = fields[11]        # Subject Strand (plus/minus)
        evalue = float(fields[12])  # Expect Value
        bitscore = float(fields[13])# Bit Score
        
        # Campos opcionais
        qlen = int(fields[14]) if len(fields) > 14 and fields[14] != '' else None    # Query sequence length
        slen = int(fields[15]) if len(fields) > 15 and fields[15] != '' else None    # Subject sequence length
        sframe = fields[16] if len(fields) > 16 and fields[16] != '' else None       # Subject frame
        
        return {
            'qseqid': qseqid,
            'sseqid': sseqid,
            'pident': pident,
            'qcovs': qcovs,
            'length': length,
            'mismatch': mismatch,
            'gapopen': gapopen,
            'qstart': qstart,
            'qend': qend,
            'sstart': sstart,
            'send': send,
            'sstrand': sstrand,
            'evalue': evalue,
            'bitscore': bitscore,
            'qlen': qlen,
            'slen': slen,
            'sframe': sframe
        }
        
    except (ValueError, IndexError) as e:
        print(f"Erro ao processar linha: {line[:100]}...")
        print(f"Erro específico: {e}")
        print(f"Número de campos encontrados: {len(fields)}")
        return None

def determine_strand(strand_info):
    """
    Determina a orientação (+/-) baseada na informação de strand
    """
    if isinstance(strand_info, str):
        strand_lower = strand_info.lower()
        if 'minus' in strand_lower or strand_lower == '-':
            return '-'
        else:
            return '+'
    return '+'

def blast_to_gff(blast_file, output_file=None, source="BLASTn"):
    """
    Converte arquivo BLAST para formato GFF3
    """
    if output_file is None:
        output_file = blast_file.replace('.txt', '.gff')
        if output_file == blast_file:
            output_file = blast_file + '.gff'
    
    # Se output_file for um diretório, criar nome do arquivo
    if os.path.isdir(output_file):
        base_name = os.path.basename(blast_file)
        name_without_ext = os.path.splitext(base_name)[0]
        output_file = os.path.join(output_file, f"{name_without_ext}.gff")
    
    # Criar diretório de saída se não existir
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    
    try:
        with open(blast_file, 'r') as f_in, open(output_file, 'w') as f_out:
            # Escreve cabeçalho GFF3
            f_out.write("##gff-version 3\n")
            
            line_count = 0
            converted_count = 0
            
            for line in f_in:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                line_count += 1
                blast_data = parse_blast_line(line)
                
                if blast_data is None:
                    print(f"Aviso: Linha {line_count} não pôde ser processada")
                    continue
                
                # Determina coordenadas no query (GFF usa coordenadas 1-based)
                # Usar coordenadas do query (qstart e qend) para posicionamento no genoma de referência
                start = min(blast_data['qstart'], blast_data['qend'])
                end = max(blast_data['qstart'], blast_data['qend'])
                
                # Determina strand baseado na orientação no subject
                strand = determine_strand(blast_data['sstrand'])
                
                # Campos do GFF3
                seqid = blast_data['qseqid']  # Sequência de referência (query)
                source_field = source
                feature_type = blast_data['sseqid']  # Nome do elemento encontrado (subject)
                score = blast_data['bitscore'] if blast_data['bitscore'] > 0 else '.'
                phase = '.'  # Não aplicável para matches
                
                # Atributos no formato GFF3
                attributes = []
                attributes.append(f"ID=blast_hit_{converted_count + 1}")
                attributes.append(f"Name={blast_data['sseqid']}")
                attributes.append(f"Target={blast_data['sseqid']} {blast_data['sstart']} {blast_data['send']}")
                attributes.append(f"identity={blast_data['pident']:.2f}")
                attributes.append(f"query_coverage={blast_data['qcovs']:.2f}")
                attributes.append(f"evalue={blast_data['evalue']}")
                attributes.append(f"bitscore={blast_data['bitscore']}")
                attributes.append(f"alignment_length={blast_data['length']}")
                attributes.append(f"mismatches={blast_data['mismatch']}")
                attributes.append(f"gap_openings={blast_data['gapopen']}")
                
                # Adiciona campos opcionais se disponíveis
                if blast_data['qlen'] is not None:
                    attributes.append(f"query_length={blast_data['qlen']}")
                if blast_data['slen'] is not None:
                    attributes.append(f"subject_length={blast_data['slen']}")
                if blast_data['sframe'] is not None:
                    attributes.append(f"subject_frame={blast_data['sframe']}")
                
                attr_string = ';'.join(attributes)
                
                # Linha GFF3
                gff_line = f"{seqid}\t{source_field}\t{feature_type}\t{start}\t{end}\t{score}\t{strand}\t{phase}\t{attr_string}\n"
                f_out.write(gff_line)
                
                converted_count += 1
            
            print(f"Conversão concluída:")
            print(f"  Arquivo de entrada: {blast_file}")
            print(f"  Arquivo de saída: {output_file}")
            print(f"  Linhas processadas: {line_count}")
            print(f"  Hits convertidos: {converted_count}")
            
    except FileNotFoundError:
        print(f"Erro: Arquivo {blast_file} não encontrado")
        return False
    except Exception as e:
        print(f"Erro durante a conversão: {e}")
        print(f"Tipo do erro: {type(e).__name__}")
        import traceback
        traceback.print_exc()
        return False
    
    return True

def convert_directory(input_dir, output_dir=None, file_pattern="*.txt"):
    """
    Converte todos os arquivos BLAST em um diretório
    """
    input_path = Path(input_dir)
    
    if not input_path.exists():
        print(f"Erro: Diretório {input_dir} não existe")
        return
    
    if output_dir:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
    else:
        output_path = input_path
    
    blast_files = list(input_path.glob(file_pattern))
    
    if not blast_files:
        print(f"Nenhum arquivo encontrado com padrão {file_pattern} em {input_dir}")
        return
    
    print(f"Encontrados {len(blast_files)} arquivos para conversão")
    
    successful = 0
    for blast_file in blast_files:
        print(f"\nProcessando: {blast_file.name}")
        
        if output_dir:
            output_file = output_path / f"{blast_file.stem}.gff"
        else:
            output_file = blast_file.with_suffix('.gff')
        
        if blast_to_gff(str(blast_file), str(output_file)):
            successful += 1
    
    print(f"\nResumo: {successful}/{len(blast_files)} arquivos convertidos com sucesso")

def main():
    parser = argparse.ArgumentParser(
        description="Converte arquivos de saída do BLASTn para formato GFF3",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Exemplos de uso:

1. Converter um arquivo específico (salva no mesmo diretório):
   python blast_to_gff.py arquivo.txt

2. Converter especificando arquivo de saída:
   python blast_to_gff.py arquivo.txt -o /caminho/completo/resultado.gff

3. Converter especificando apenas o nome do arquivo de saída:
   python blast_to_gff.py arquivo.txt -o meu_resultado.gff

4. Converter todos os arquivos de um diretório (salva no mesmo diretório):
   python blast_to_gff.py -d /caminho/para/diretorio/

5. Converter todos os arquivos para um diretório diferente:
   python blast_to_gff.py -d /input/dir/ -od /output/dir/

6. Converter com padrão específico para diretório de saída:
   python blast_to_gff.py -d /input/ -od /output/ -p "GCF_*.txt"
        """
    )
    
    parser.add_argument('input', nargs='?', help='Arquivo BLAST de entrada (ou use -d para diretório)')
    parser.add_argument('-o', '--output', help='Arquivo GFF de saída')
    parser.add_argument('-d', '--directory', help='Diretório contendo arquivos BLAST')
    parser.add_argument('-od', '--output-dir', help='Diretório de saída (para modo diretório)')
    parser.add_argument('-p', '--pattern', default='*.txt', help='Padrão de arquivos (default: *.txt)')
    parser.add_argument('-s', '--source', default='BLASTn', help='Nome da fonte para GFF (default: BLASTn)')
    
    args = parser.parse_args()
    
    if args.directory:
        # Modo diretório
        convert_directory(args.directory, args.output_dir, args.pattern)
    elif args.input:
        # Modo arquivo único
        blast_to_gff(args.input, args.output, args.source)
    else:
        parser.print_help()
        sys.exit(1)

if __name__ == "__main__":
    main()

###################################################################################################################################################

# ctrl + o (salvar) -> enter -> ctrl + x (sair)
# Agora e só rodar o script para converter os arquivos .txt gerados pelo BLAST em .gff

1. Converter um arquivo específico (salva no mesmo diretório):
python blast_to_gff.py arquivo.txt

2. Converter especificando arquivo de saída:
python blast_to_gff.py arquivo.txt -o /caminho/completo/resultado.gff

3. Converter especificando apenas o nome do arquivo de saída:
python blast_to_gff.py arquivo.txt -o meu_resultado.gff

4. Converter todos os arquivos de um diretório (salva no mesmo diretório):
python blast_to_gff.py -d /caminho/para/diretorio/

5. Converter todos os arquivos para um diretório diferente:
python blast_to_gff.py -d /input/dir/ -od /output/dir/

6. Converter com padrão específico para diretório de saída:
python blast_to_gff.py -d /input/ -od /output/ -p "GCF_*.txt"